# Урок 18. Контрольная работа № 2

9 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [← Урок 17](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-17.ipynb) · [Урок 19 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-19.ipynb)

---

Разбор ошибок. Самостоятельная работа: функции, словари, файлы.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 9А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
for адрес in ("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py",
              "https://cdn.jsdelivr.net/gh/LeksssGray/informatika-7-11@main/lib/schoolinf.py"):
    try:
        urllib.request.urlretrieve(адрес, "schoolinf.py")
        break
    except Exception as ошибка:
        print("⚠️ Не скачалась библиотека проверки:", ошибка)
else:
    raise SystemExit("Проверь интернет и запусти эту ячейку ещё раз. "
                     "Не помогло — покажи это сообщение учителю.")
import schoolinf as si
importlib.reload(si)
si.start(lesson="09-18", name=ФИО, klass=Класс)

## Где обычно ошибаются

Четверть была про инструменты: функции, рекурсия, строки, словари,
файлы. Соберём ошибки, которые встречались чаще всего, а потом
напишем контрольную.

### Ошибка 1. Функция печатает вместо того, чтобы возвращать

`print` показывает значение человеку, `return` отдаёт его программе.
Функция без `return` возвращает `None`, и автопроверка честно скажет,
что получила не то. Правило: считает — возвращает, выводит — печатает.

### Ошибка 2. Рекурсия без базового случая

Или базовый случай есть, но до него не доходят: `сумма(n - 2)` при
нечётном `n` проскакивает мимо нуля. Проверяйте, что шаг обязательно
приводит к базовому случаю.

### Ошибка 3. Строку пытаются изменить

`строка[0] = "А"` — ошибка. Строки неизменяемы, все методы возвращают
**новую** строку, и результат нужно куда-то присвоить:
`строка = строка.replace("а", "о")`.

### Ошибка 4. Обращение к отсутствующему ключу

`d[ключ]` при отсутствующем ключе — `KeyError`. В накоплении частот
всегда `d.get(ключ, 0) + 1`.

### Ошибка 5. Из файла пришли строки

`"10" < "9"` — истина, потому что сравниваются буквы. Превращайте
в число сразу: `int(строка.strip())`.

### Ошибка 6. Забыт `strip()` или пустая строка в конце файла

Два симптома одной беды: то `int("")` падает, то сравнение строк
не срабатывает из-за невидимого `\n`.

### Ошибка 7. Режим `"w"` вместо `"r"`

Открыли файл на запись, чтобы прочитать, — и данных больше нет.
Восстановить их невозможно.

### Чек-лист перед сдачей

1. У каждой функции есть `return`?
2. Рекурсия точно доходит до базового случая?
3. Всё, что пришло из файла, превращено в числа, где нужно?
4. Словарь заполняется через `.get()`?
5. Файл открыт в правильном режиме?

## Разбираем сложное

### Пример 1. Одна задача — три инструмента сразу

> В файле построчно записаны слова. Найдите слово, которое встречается
> чаще всех, и запишите отчёт.

Здесь встречаются все темы четверти: файл, словарь, строки и функции.

In [ ]:
with open("слова.txt", "w", encoding="utf-8") as файл:
    файл.write("кот\nпёс\nкот\nёж\nкот\nпёс\n")


def частоты_из_файла(имя_файла):
    частоты = {}
    with open(имя_файла, encoding="utf-8") as файл:
        for строка in файл:
            слово = строка.strip().lower()
            if слово:
                частоты[слово] = частоты.get(слово, 0) + 1
    return частоты


def самое_частое(частоты):
    return max(частоты, key=lambda слово: частоты[слово])


частоты = частоты_из_файла("слова.txt")
победитель = самое_частое(частоты)

print(частоты)
print(f"Чаще всех: {победитель} ({частоты[победитель]} раза)")

Обратите внимание на разделение: одна функция читает файл, другая
ищет максимум. Вторую можно проверить без всякого файла — просто
передав ей готовый словарь. Это и есть смысл разбиения на функции.

### Пример 2. Рекурсия там, где цикл неудобен

Дерево папок задано словарём «папка → что внутри». Посчитаем общий
размер, спускаясь по вложенности.

In [ ]:
дерево = {
    "/": ["документы", "фото", "заметка.txt"],
    "документы": ["отчёт.docx", "смета.xlsx"],
    "фото": ["лето", "кот.jpg"],
    "лето": ["море.jpg", "горы.jpg"],
}

размеры = {
    "заметка.txt": 2, "отчёт.docx": 40, "смета.xlsx": 25,
    "кот.jpg": 300, "море.jpg": 500, "горы.jpg": 450,
}


def размер(узел):
    if узел in размеры:          # это файл — базовый случай
        return размеры[узел]
    итог = 0
    for внутри in дерево[узел]:  # это папка — спускаемся глубже
        итог += размер(внутри)
    return итог


for папка in ("лето", "фото", "документы", "/"):
    print(f"{папка:<11} {размер(папка)} КБ")

Циклом такое пишется заметно сложнее: заранее неизвестно, насколько
глубоко вложены папки. Рекурсия справляется с любой глубиной,
потому что не знает про неё вовсе — просто спрашивает соседей.

## Контрольная работа

Работа из двух частей. Сначала восемь задач по материалу
четверти — они у всех одинаковые. Потом тест: у каждого свой
вариант из восьми заданий, у соседа по парте задания другие.

### Задача 1. Функция с несколькими значениями

Функция возвращает кортеж `(количество, сумма, максимум)` для списка
чисел. Список непустой.

In [ ]:
def сводка_чисел(числа):
    return ...

In [ ]:
si.check("1", сводка_чисел, [
    ([1, 2, 3], (3, 6, 3)),
    ([7], (1, 7, 7)),
    ([-5, -2], (2, -7, -2)),
])

### Задача 2. Рекурсия

Рекурсивная функция: произведение чисел от 1 до n. Для n ≤ 0
возвращайте 1. Циклы использовать нельзя.

In [ ]:
def произведение(n):
    return ...

In [ ]:
si.check("2", произведение, [
    (5, 120),
    (1, 1),
    (0, 1),
    (10, 3628800),
])

### Задача 3. Рекурсия по цифрам

Рекурсивная функция: количество чётных цифр в натуральном числе.
`чётных_цифр(1234)` → 2, `чётных_цифр(0)` → 1.

In [ ]:
def чётных_цифр(n):
    return ...

In [ ]:
si.check("3", чётных_цифр, [
    (1234, 2),
    (0, 1),
    (13579, 0),
    (2468, 4),
])

### Задача 4. Строки

Функция возвращает список слов текста длиннее заданного числа букв.
Порядок сохраняется, регистр не меняется, знаков препинания нет.

In [ ]:
def длинные_слова(текст, минимум):
    return ...

In [ ]:
si.check("4", длинные_слова, [
    (("мама мыла раму", 3), ["мама", "мыла", "раму"]),
    (("а бб ввв гггг", 2), ["ввв", "гггг"]),
    (("", 1), []),
])

### Задача 5. Словарь

Функция получает список оценок и возвращает словарь
«оценка → сколько раз встретилась». Оценок, которых нет, в словаре
быть не должно.

In [ ]:
def распределение(оценки):
    return ...

In [ ]:
si.check("5", распределение, [
    ([5, 4, 5, 3], {5: 2, 4: 1, 3: 1}),
    ([], {}),
    ([2, 2, 2], {2: 3}),
])

### Задача 6. Файл: запись и чтение

Функция записывает числа в файл (по одному в строке) и возвращает
их сумму.

In [ ]:
def сохранить_числа(имя_файла, числа):
    return ...

In [ ]:
si.check("6", сохранить_числа, [
    (("кр1.txt", [1, 2, 3]), 6),
    (("кр2.txt", [10, 20]), 30),
    (("кр3.txt", []), 0),
])

### Задача 7. Файл: обработка

В файле по одному целому числу в строке. Функция возвращает кортеж
`(количество, максимум)`. Для пустого файла — `(0, 0)`.

In [ ]:
def разбор_чисел(имя_файла):
    return ...

In [ ]:
si.check("7", разбор_чисел, [
    ("кр1.txt", (3, 3)),
    ("кр2.txt", (2, 20)),
    ("кр3.txt", (0, 0)),
])

### Задача 8. Файл с полями

В файле строки вида `товар;количество;цена`. Функция возвращает общую
стоимость всех покупок (количество × цена, сложить по всем строкам).

In [ ]:
def стоимость(имя_файла):
    return ...

In [ ]:
with open("покупки.txt", "w", encoding="utf-8") as файл:
    файл.write("хлеб;2;40\nмолоко;1;75\nяблоки;3;120\n")

si.check("8", стоимость, [
    ("покупки.txt", 515),
])

### Контрольный тест

Запусти ячейку ниже — получишь свой вариант из 8 заданий. Он собран из общего банка вопросов, и у соседа по парте почти наверняка будет другой.

Вариант не меняется от запуска к запуску: можно перезапускать ячейку сколько угодно, задания останутся теми же.

Правильных ответов в ноутбуке нет — только их хеши, так что подсмотреть не выйдет. Разбор работы будет у учителя.

In [ ]:
#@title 📋 Мой вариант { display-mode: "form" }
#@markdown Запусти ячейку — появятся твои задания.
# автоген: банк вопросов
БАНК = [
    {'id': 'q01', 'текст': 'def f(x): return x * 2 + 1. Чему равно f(f(3))?', 'хеш': 'e629fa6598d73276', 'варианты': None},
    {'id': 'q02', 'текст': 'def f(a, b=10): return a - b. Чему равно f(25)?', 'хеш': 'e629fa6598d73276', 'варианты': None},
    {'id': 'q03', 'текст': 'Что возвращает функция, в которой нет return?', 'хеш': '30a0924a3360be68', 'варианты': ['None', '0', 'пустую строку']},
    {'id': 'q04', 'текст': 'def g(n): return n, n * n. Что вернёт g(4)?', 'хеш': '6a8132b8d7509faf', 'варианты': ['(4, 16)', '[4, 16]', '4 16']},
    {'id': 'q05', 'текст': 'def f(n): if n == 0: return 0; иначе return n + f(n - 1). Чему равно f(6)?', 'хеш': '6f4b6612125fb3a0', 'варианты': None},
    {'id': 'q06', 'текст': 'def f(n): if n <= 1: return 1; иначе return n * f(n - 1). Чему равно f(5)?', 'хеш': '2abaca4911e68fa9', 'варианты': None},
    {'id': 'q07', 'текст': 'def F(n): if n < 2: return n; иначе return F(n - 1) + F(n - 2). Чему равно F(10)?', 'хеш': '02d20bbd7e394ad5', 'варианты': None},
    {'id': 'q08', 'текст': 'def F(n): if n < 2: return n; иначе return F(n - 1) + F(n - 2). Сколько всего вызовов F произойдёт при вычислении F(4), считая самый первый?', 'хеш': '19581e27de7ced00', 'варианты': None},
    {'id': 'q09', 'текст': 'Как называется случай, при котором рекурсивная функция больше не вызывает себя?', 'хеш': 'cf1c29e96ab00b73', 'варианты': ['базовый случай', 'шаг рекурсии', 'глубина рекурсии']},
    {'id': 'q10', 'текст': 'def s(n): if n < 10: return n; иначе return n % 10 + s(n // 10). Чему равно s(4096)?', 'хеш': '9400f1b21cb527d7', 'варианты': None},
    {'id': 'q11', 'текст': 'Чему равно len("информатика")?', 'хеш': '4fc82b26aecb47d2', 'варианты': None},
    {'id': 'q12', 'текст': 'Чему равно "мама мыла раму".count("ма")?', 'хеш': 'd4735e3a265e16ee', 'варианты': None},
    {'id': 'q13', 'текст': 'Что получится в результате "Python"[1:4]? Ответ запишите без кавычек.', 'хеш': 'e996464aff3210bc', 'варианты': None},
    {'id': 'q14', 'текст': 'Сколько элементов в списке "a,b,,c".split(",")?', 'хеш': '4b227777d4dd1fc6', 'варианты': None},
    {'id': 'q15', 'текст': 'Сколько символов в строке "  привет  ".strip()?', 'хеш': 'e7f6c011776e8db7', 'варианты': None},
    {'id': 'q16', 'текст': 'Что будет, если выполнить s = "кот", а затем s[0] = "К"?', 'хеш': '2757fbfee5393ab2', 'варианты': ['ошибка', 'Кот', 'Ккот']},
    {'id': 'q17', 'текст': 'd = {"а": 1, "б": 2}; затем d["в"] = 3 и d["а"] += 10. Чему равна сумма всех значений словаря?', 'хеш': 'b17ef6d19c7a5b1e', 'варианты': None},
    {'id': 'q18', 'текст': 'd = {"x": 5}. Что вернёт d.get("y", 0)?', 'хеш': '5feceb66ffc86f38', 'варианты': None},
    {'id': 'q19', 'текст': 'Какая ошибка возникнет при обращении d["y"], если ключа "y" в словаре нет?', 'хеш': 'f4d66d147390bcbf', 'варианты': ['KeyError', 'IndexError', 'ValueError']},
    {'id': 'q20', 'текст': 'Частоты букв слова "банан" собрали в словарь. Какое значение у ключа "а"?', 'хеш': 'd4735e3a265e16ee', 'варианты': None},
    {'id': 'q21', 'текст': 'Сколько ключей будет в словаре частот букв слова "колокол"?', 'хеш': '4e07408562bedb8b', 'варианты': None},
    {'id': 'q22', 'текст': 'В каком режиме open стирает старое содержимое файла и пишет заново?', 'хеш': '7265b875feb0d173', 'варианты': ['r', 'w', 'a']},
    {'id': 'q23', 'текст': 'В файле три строки: 5, 8 и 13. Чему будет равна s после кода: s = 0; for line in f: s += int(line)?', 'хеш': '5f9c4ab08cac7457', 'варианты': None},
    {'id': 'q24', 'текст': 'Строку файла "Петров;9Б;74" разбили: поля = строка.split(";"). Чему равно int(поля[2]) + 1?', 'хеш': 'f369cb89fc627e66', 'варианты': None},
]

si.тест("к2", БАНК, сколько=8)

In [ ]:
#@title ✍️ Мои ответы { display-mode: "form" }
#@markdown Впиши ответ на каждое задание своего варианта.
Задание_1 = "" #@param {type:"string"}
Задание_2 = "" #@param {type:"string"}
Задание_3 = "" #@param {type:"string"}
Задание_4 = "" #@param {type:"string"}
Задание_5 = "" #@param {type:"string"}
Задание_6 = "" #@param {type:"string"}
Задание_7 = "" #@param {type:"string"}
Задание_8 = "" #@param {type:"string"}

# автоген: банк вопросов
si.ответы("к2", [Задание_1, Задание_2, Задание_3, Задание_4, Задание_5, Задание_6, Задание_7, Задание_8])

## Домашнее задание на каникулы

### Домашнее задание. Анализатор файла

Функция по файлу вида `товар;количество;цена` возвращает список
из четырёх значений:

1. количество строк;
2. общую стоимость;
3. название самого дорогого товара (по цене за штуку);
4. количество товаров дороже 100 за штуку.

Если максимальную цену делят несколько товаров, берите первый.

In [ ]:
def анализ_покупок(имя_файла):
    return ...

In [ ]:
si.check("дз", анализ_покупок, [
    ("покупки.txt", [3, 515, "яблоки", 1]),
])

---

### Итоги четверти

За вторую четверть вы научились разбивать программу на функции
и возвращать из них несколько значений, писать рекурсию, обрабатывать
текст, хранить данные в словарях и работать с файлами — от чтения
до отчёта.

Этого набора достаточно, чтобы решать настоящие задачи с данными,
и именно им мы займёмся в третьей четверти: электронные таблицы
на тысячи строк, диаграммы, базы данных и запросы к ним.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 17](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-17.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 19 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-19.ipynb)